# Prompt Chaining

## Introduction

![alt text](prompt_chaning_1.excalidraw.png)

In [2]:
# import your LLM model
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.1:8b')
results = llm.invoke('Hi')
print(results)

content="How's it going? Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-11-29T05:14:37.5675866Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1691892300, 'load_duration': 34679900, 'prompt_eval_count': 11, 'prompt_eval_duration': 422639600, 'eval_count': 21, 'eval_duration': 1233879800, 'model_name': 'llama3.1:8b'} id='run--4c799632-0f1d-4cdd-8a7c-77fca79fdacf-0' usage_metadata={'input_tokens': 11, 'output_tokens': 21, 'total_tokens': 32}


In [3]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display,Markdown
from langchain_core.runnables.graph import MermaidDrawMethod
import nest_asyncio
import os
nest_asyncio.apply()
os.environ["PYPPETEER_CHROMIUM_REVISION"] = "1181217"
os.environ["PYPPETEER_BROWSER_EXECUTABLE_PATH"] = r"C:\Users\Hanif\AppData\Local\pyppeteer\pyppeteer\chrome-win\chrome.exe"

# Define graph state
class State(TypedDict):
    topic: str 
    story: str
    criteria_story : str
    improved_story:str
    final_story:str

# Nodes

def generate_story(state:State):
    msg = llm.invoke(f"Write a one sentence story with premise about {state['topic']}")
    return {"story":msg.content}

def check_criteria(state: State):
    story_text = state['improved_story'] if state.get('improved_story') else state['story']
    msg = llm.invoke(
        f"Does the story contain three characters or more? "
        f"Answer with Failed or Passed only, don't use another word. Story: {story_text}"
    )
    return {"criteria_story": msg.content}
def improve_story(state:State):
    msg = llm.invoke(f"Improve the story by three characters or more. Story: {state['story']}")
    return {"improved_story":msg.content}

def final_user_story(state:State):
    final_story = state.get('improved_story') or state.get('story', '')
    return {"final_story": final_story}

In [4]:
graph = StateGraph(State)
# Define nodes
graph.add_node("generate",generate_story)
graph.add_node("criteria",check_criteria)
graph.add_node("improve",improve_story)
graph.add_node("final",final_user_story)

# Define edges
graph.add_edge(START,"generate")
graph.add_edge("generate", "criteria")
graph.add_conditional_edges(
    "criteria",
    lambda state: "Passed" if "Passed" in state["criteria_story"] else "Failed",
    {"Passed": "final", "Failed": "improve"}
)


graph.add_edge("improve", "criteria")



graph.add_edge("final",END) 

# Compile the graph
compiled_graph = graph.compile()

# Visualize the graph

mermaid_code = compiled_graph.get_graph().draw_mermaid()
with open("graph.mmd", "w") as f:
    f.write(mermaid_code)

In [5]:
# Stream to monitoring graph execution
config = {'configurable': {'thread_id': '2'}}
state={"topic":"Corporate success"}
for chunk in compiled_graph.stream(state,config,stream_mode='updates'):
    print(chunk)

{'generate': {'story': "As she stood on the rooftop, champagne glass in hand, Emma gazed out at the glittering city skyline and thought back to the early days of her startup, when the only thing more uncertain than their future was whether they'd ever make payroll."}}
{'criteria': {'criteria_story': 'Passed'}}
{'final': {'final_story': "As she stood on the rooftop, champagne glass in hand, Emma gazed out at the glittering city skyline and thought back to the early days of her startup, when the only thing more uncertain than their future was whether they'd ever make payroll."}}


In [6]:
state={"topic":"How to be successful in corporate world?"}
result = compiled_graph.invoke(state)
print(result)

{'topic': 'How to be successful in corporate world?', 'story': "After spending years climbing the corporate ladder, Maya finally realized that her success was not due to her extensive networking or Harvard business degree, but rather her ability to listen more than she spoke and her unwavering commitment to understanding what truly drove her team's work.", 'criteria_story': 'Passed', 'final_story': "After spending years climbing the corporate ladder, Maya finally realized that her success was not due to her extensive networking or Harvard business degree, but rather her ability to listen more than she spoke and her unwavering commitment to understanding what truly drove her team's work."}


In [7]:
print(f"initial story : {result['story']}")
print(f"final story : {result['final_story']}")

initial story : After spending years climbing the corporate ladder, Maya finally realized that her success was not due to her extensive networking or Harvard business degree, but rather her ability to listen more than she spoke and her unwavering commitment to understanding what truly drove her team's work.
final story : After spending years climbing the corporate ladder, Maya finally realized that her success was not due to her extensive networking or Harvard business degree, but rather her ability to listen more than she spoke and her unwavering commitment to understanding what truly drove her team's work.


# Evaluator Optimizer

generate response while another evaluation and feedback in loop